# 02 Hyperparameter tuning: Single-stage Lasso

Part of `03b_training_pipeline_model/single_stage_lasso`; reads `feature_config.py` (written by notebook 01) and writes `model_config.py`.

One Lasso on log rent using the static features, the selected temporal features of the flat's month and **all the interactions** between the numeric
static features and the temporal features. The columns are standardised on the training rows, so the penalty treats all columns alike. The model is fit on
the train split (2021-01 to 2024-09) and scored on the validation split (2024-10 to 2025-03) for three training windows: the **full history** (from
2021-01), **2023 onward** (from 2023-01) and **after May 2023** (from 2023-06 only). The penalty is chosen per window by **validation log RMSE**.

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

SHARED_DIR = '/Users/keltonn/Desktop/Projects/housing_price_regressor/03b_training_pipeline_model'
FOLDER = f'{SHARED_DIR}/single_stage_lasso'
sys.path.insert(0, SHARED_DIR)      # pipeline_utils.py, shared by every model folder
sys.path.insert(0, FOLDER)          # this folder's feature_config.py and model_config.py
import pprint
from sklearn.linear_model import Lasso, Ridge, lasso_path
from sklearn.preprocessing import StandardScaler
import feature_config
import pipeline_utils as pu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

ALPHAS = [3e-2, 1e-2, 5e-3, 3e-3, 2e-3, 1e-3, 5e-4, 3e-4, 1e-4, 3e-5, 1e-5]     # on standardised columns
FAMILY = 'single_stage_lasso'

## 1. Data

In [2]:
frames, numeric, onehot = pu.load_static()
temporal = pu.load_temporal()
n_num = len(numeric)
tr, va = frames['train'], frames['validation']
X_tr, y_tr, m_tr = pu.design(tr, numeric, onehot), pu.log_rent(tr), tr['month'].to_numpy()
X_va, y_va, m_va = pu.design(va, numeric, onehot), pu.log_rent(va), va['month'].to_numpy()
val_months = np.arange(pu.VALIDATION_FIRST, pu.VALIDATION_LAST + 1)
print(f'train {X_tr.shape}, validation {X_va.shape}')

train (132398, 237), validation (16961, 237)


## 2. Fit every penalty (`lasso_path`, warm starts)

In [3]:
def evaluate(window):
    start = pu.WINDOW_START[window]
    months_fit = np.arange(start, pu.TRAIN_LAST + 1)
    s1_months = months_fit[months_fit >= 1]
    selected = feature_config.SELECTED_TEMPORAL_FEATURES[f'{window}/train']
    Z = pu.zscore(temporal, selected, s1_months)
    rows = np.flatnonzero(m_tr >= start)
    D = pu.single_stage_design(X_tr[rows], n_num, Z, m_tr[rows])
    scaler = StandardScaler().fit(D)
    Ds, y_mean = scaler.transform(D), y_tr[rows].mean()
    Dv = scaler.transform(pu.single_stage_design(X_va, n_num, Z, m_va))
    print(f'{window}: {D.shape[1]} columns')
    lasso_alphas, coefs, _ = lasso_path(Ds, y_tr[rows] - y_mean, alphas=sorted(ALPHAS, reverse=True), precompute=True, max_iter=5000)
    penalties, used = [float(a) for a in lasso_alphas], [int((coefs[:, i] != 0).sum()) for i in range(coefs.shape[1])]
    return [{'window': window, 'family': FAMILY, 'alpha': penalties[i], 'features_used': used[i], **pu.score(y_va, Dv @ coefs[:, i] + y_mean)}
            for i in range(len(penalties))]


results = pd.DataFrame([row for window in pu.WINDOW_START for row in evaluate(window)])
display(results.pivot(index='alpha', columns='window', values='log_rmse').sort_index(ascending=False).round(4).rename_axis('validation log RMSE by penalty'))

full_history: 557 columns


from_2023: 637 columns


from_2023_06: 557 columns


window,from_2023,from_2023_06,full_history
validation log RMSE by penalty,,,
0.03000,0.1996,0.1981,0.2111
0.01000,0.1857,0.1856,0.1963
0.00500,0.1832,0.1819,0.1935
0.00300,0.1821,0.1803,0.1923
0.00200,0.1815,0.1793,0.1893
0.00100,0.1812,0.1788,0.1844
0.00050,0.1813,0.1789,0.1824
0.00030,0.1814,0.1791,0.1818
0.00010,0.1841,0.1797,0.1815


## 3. Best penalty per window

In [4]:
best = results.loc[results.groupby('window')['log_rmse'].idxmin()].set_index('window')
view = best[['log_r2', 'dollar_r2', 'log_rmse', 'dollar_rmse']].copy()
view['settings'] = best.apply(lambda r: f"alpha={r['alpha']:g}, {int(r['features_used'])} of the columns used", axis=1)
display(view.sort_values('log_rmse').round(4))

,log_r2,dollar_r2,log_rmse,dollar_rmse,settings
window,,,,,
from_2023_06,0.3833,0.4573,0.1788,472.6818,"alpha=0.001, 166 of the columns used"
from_2023,0.3668,0.4216,0.1812,488.0098,"alpha=0.001, 162 of the columns used"
full_history,0.3650,0.4204,0.1815,488.4929,"alpha=0.0001, 285 of the columns used"


## 4. Write `model_config.py`

In [5]:
def plain(params):
    return {k: (float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) and not isinstance(v, bool) else v)
            for k, v in params.items()}


def metrics(row):
    return {m: round(float(row[m]), 5) for m in ('log_r2', 'dollar_r2', 'log_rmse', 'dollar_rmse')}


setups = {}
for window in pu.WINDOW_START:
    r = best.loc[window]
    setups[f'{window}/{FAMILY}'] = {'lasso_alpha': float(r['alpha']), 'validation': metrics(r)}
text = ('"""Best setup per training window for this model folder, chosen on the validation set (lowest log RMSE) by 02_tune_model.ipynb.\n'
        'Rerun that notebook to update it."""\n\nBEST_SETUPS = ' + pprint.pformat(setups, width=140, sort_dicts=False) + '\n')
with open(f'{FOLDER}/model_config.py', 'w') as f:
    f.write(text)
print(text)

"""Best setup per training window for this model folder, chosen on the validation set (lowest log RMSE) by 02_tune_model.ipynb.
Rerun that notebook to update it."""

BEST_SETUPS = {'full_history/single_stage_lasso': {'lasso_alpha': 0.0001,
                                     'validation': {'log_r2': 0.36496,
                                                    'dollar_r2': 0.42041,
                                                    'log_rmse': 0.18147,
                                                    'dollar_rmse': 488.49287}},
 'from_2023/single_stage_lasso': {'lasso_alpha': 0.001,
                                  'validation': {'log_r2': 0.36682, 'dollar_r2': 0.42156, 'log_rmse': 0.1812, 'dollar_rmse': 488.0098}},
 'from_2023_06/single_stage_lasso': {'lasso_alpha': 0.001,
                                     'validation': {'log_r2': 0.38331,
                                                    'dollar_r2': 0.45732,
                                                    'log_rmse': 0